# 11 — Measure key events with canonical arrival windows

Measure named-event pressure amplitudes and export the exact source times,
predicted reference-channel arrivals, waveform windows, baselines and signal
intervals used by downstream manuscript and figure notebooks.


Positive, negative, peak-to-peak and 1-km-reduced pressure are
measured from the canonical Notebook 02 moving-median-baseline-corrected pressure
Stream. Each manually reviewed window is expressed relative to the predicted DD2
arrival calculated from the canonical source time and a declared 351 m s⁻¹
propagation speed. Short local baseline intervals remove residual constant offsets.


This compact notebook provides the named-event measurements used
in manuscript text and figure annotations. It remains separate from Notebook 10
because these windows are manually reviewed for a few physically important
arrivals rather than derived from the full event catalogue.


In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import Stream, UTCDateTime

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from modules import project_config as config

DERIVED_DIR = config.NB110_DIR
FIGURE_DIR = config.NB110_DIR

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})


In [2]:
from obspy import read
from modules.event_measurements import (
    measure_reduced_pressures_in_window,
    pressure_results_for_paper,
)

ANALYSIS_CONFIG_FILE = (
    config.NB010_DIR / "analysis_configuration.json"
)

if not ANALYSIS_CONFIG_FILE.exists():
    raise FileNotFoundError(ANALYSIS_CONFIG_FILE)

analysis_config = json.loads(
    ANALYSIS_CONFIG_FILE.read_text()
)

GEOMETRY_FILE = Path(
    analysis_config["geometry_file"]
).expanduser()

if not GEOMETRY_FILE.exists():
    raise FileNotFoundError(
        "Geometry product recorded by Notebook 02 was not found: "
        f"{GEOMETRY_FILE}"
    )

baseline_products = analysis_config.get(
    "baseline_removed_streams",
    {},
)
BASELINE_STREAM_FILE = Path(
    baseline_products.get(
        "pickle",
        config.NB010_DIR
        / "bchh_corrected_moving_median_baseline_removed.pkl",
    )
).expanduser()

if not BASELINE_STREAM_FILE.exists():
    raise FileNotFoundError(
        "Run Notebook 02 to create the baseline-corrected Stream: "
        f"{BASELINE_STREAM_FILE}"
    )

st_corr = read(
    str(BASELINE_STREAM_FILE),
    format="PICKLE",
)

geometry = pd.read_csv(GEOMETRY_FILE)
geometry["channel"] = (
    geometry["channel"]
    .astype(str)
    .str.upper()
)

pressure_channels = tuple(
    analysis_config["pressure_channels"]
)

noncanonical_channels = [
    channel
    for channel in pressure_channels
    if not channel.startswith("DD")
]
if noncanonical_channels:
    raise ValueError(
        "Notebook 02 configuration contains noncanonical pressure "
        f"channels: {noncanonical_channels}"
    )

SENSOR_DISTANCES_M = (
    geometry.loc[
        geometry["channel"].isin(pressure_channels)
    ]
    .set_index("channel")["distance_m"]
    .astype(float)
    .to_dict()
)

missing_geometry_channels = (
    set(pressure_channels) - set(SENSOR_DISTANCES_M)
)
if missing_geometry_channels:
    raise KeyError(
        "Geometry table lacks canonical pressure channels: "
        f"{sorted(missing_geometry_channels)}"
    )

stream_channels = {
    trace.stats.channel
    for trace in st_corr
}
missing_stream_channels = (
    set(pressure_channels) - stream_channels
)
if missing_stream_channels:
    raise KeyError(
        "Notebook 02 baseline Stream lacks canonical pressure "
        f"channels: {sorted(missing_stream_channels)}"
    )

print("Pressure source:", BASELINE_STREAM_FILE)
print("Geometry:", GEOMETRY_FILE)
print("Pressure channels:", pressure_channels)
print("Sensor distances (m):", SENSOR_DISTANCES_M)


REFERENCE_CHANNEL = analysis_config["infrasound_reference_channel"]
if REFERENCE_CHANNEL not in SENSOR_DISTANCES_M:
    raise KeyError(f"Reference channel {REFERENCE_CHANNEL!r} lacks a geometry distance")

ACOUSTIC_PROPAGATION_SPEED_MPS = 351.0
REFERENCE_CHANNEL_DISTANCE_M = float(SENSOR_DISTANCES_M[REFERENCE_CHANNEL])
REFERENCE_CHANNEL_TRAVEL_TIME_S = (
    REFERENCE_CHANNEL_DISTANCE_M / ACOUSTIC_PROPAGATION_SPEED_MPS
)

print(
    f"Arrival reference: {REFERENCE_CHANNEL}, "
    f"distance={REFERENCE_CHANNEL_DISTANCE_M:.3f} m, "
    f"speed={ACOUSTIC_PROPAGATION_SPEED_MPS:.1f} m/s, "
    f"travel={REFERENCE_CHANNEL_TRAVEL_TIME_S:.6f} s"
)


Pressure source: /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/falcon9-seismoacoustic-workflow-1.0.0/data/outputs/010_prepare_analysis_inputs/bchh_corrected_moving_median_baseline_removed.pkl
Geometry: /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/falcon9-seismoacoustic-workflow-1.0.0/data/outputs/010_prepare_analysis_inputs/bchh_channels.csv
Pressure channels: ('DD1', 'DD2', 'DD3')
Sensor distances (m): {'DD1': 1439.9638305217552, 'DD2': 1408.3384566901343, 'DD3': 1412.8543304792386}
Arrival reference: DD2, distance=1408.338 m, speed=351.0 m/s, travel=4.012360 s


## Measurement windows

The broad window shapes and internal baseline/signal intervals were retained from
the manually reviewed analysis. Their absolute UTC positions are now generated
from each canonical source time plus the predicted DD2 travel time, eliminating
the former `−0.08 s` adjustments and hard-coded capsule arrival times.


In [3]:
required_source_events = (
    "upper_stage",
    "lower_stage",
    "capsule_impact",
    "capsule_explosion",
)
missing_source_events = [
    key for key in required_source_events
    if key not in config.SOURCE_EVENT_TIMES
]
if missing_source_events:
    raise KeyError(
        f"SOURCE_EVENT_TIMES lacks required named events: {missing_source_events}"
    )


def canonical_arrival_window(
    event,
    source_event_key,
    window_start_offset_s,
    window_duration_s,
    baseline_start_s,
    baseline_end_s,
    signal_start_s,
    signal_end_s,
):
    source_time = UTCDateTime(config.SOURCE_EVENT_TIMES[source_event_key])
    predicted_arrival = source_time + REFERENCE_CHANNEL_TRAVEL_TIME_S
    window_start = predicted_arrival + window_start_offset_s
    return {
        "event": event,
        "source_event_key": source_event_key,
        "source_time": source_time,
        "predicted_reference_arrival": predicted_arrival,
        "window_start_offset_from_predicted_arrival_s": float(window_start_offset_s),
        "start": window_start,
        "end": window_start + window_duration_s,
        "baseline_start_s": float(baseline_start_s),
        "baseline_end_s": float(baseline_end_s),
        "signal_start_s": float(signal_start_s),
        "signal_end_s": float(signal_end_s),
    }


event_windows = [
    canonical_arrival_window(
        event="Initial second-stage failure",
        source_event_key="upper_stage",
        window_start_offset_s=-0.92,
        window_duration_s=2.00,
        baseline_start_s=0.10,
        baseline_end_s=0.80,
        signal_start_s=0.85,
        signal_end_s=1.40,
    ),
    canonical_arrival_window(
        event="Principal explosion",
        source_event_key="lower_stage",
        window_start_offset_s=-1.45,
        window_duration_s=3.00,
        baseline_start_s=0.10,
        baseline_end_s=0.75,
        signal_start_s=0.75,
        signal_end_s=2.50,
    ),
    # Begin the capsule-impact signal 0.04 s before the predicted DD2
    # arrival so that the leading compression is not clipped. Retain a
    # 0.01-s guard interval after the pre-event baseline.
    canonical_arrival_window(
        event="Capsule pulse 1",
        source_event_key="capsule_impact",
        window_start_offset_s=-0.12,
        window_duration_s=0.45,
        baseline_start_s=0.00,
        baseline_end_s=0.07,
        signal_start_s=0.08,
        signal_end_s=0.45,
    ),
    canonical_arrival_window(
        event="Capsule pulse 2",
        source_event_key="capsule_explosion",
        window_start_offset_s=-0.13,
        window_duration_s=0.40,
        baseline_start_s=0.00,
        baseline_end_s=0.10,
        signal_start_s=0.10,
        signal_end_s=0.40,
    ),
]

display(pd.DataFrame([
    {
        "event": spec["event"],
        "source_event_key": spec["source_event_key"],
        "source_time_utc": spec["source_time"].isoformat(),
        "predicted_reference_arrival_utc": spec["predicted_reference_arrival"].isoformat(),
        "window_start_utc": spec["start"].isoformat(),
        "window_end_utc": spec["end"].isoformat(),
    }
    for spec in event_windows
]))


,event,source_event_key,source_time_utc,predicted_reference_arrival_utc,window_start_utc,window_end_utc
0,Initial second-stage failure,upper_stage,2016-09-01T13:07:11.913000,2016-09-01T13:07:15.925360,2016-09-01T13:07:15.005360,2016-09-01T13:07:17.005360
1,Principal explosion,lower_stage,2016-09-01T13:07:15.513600,2016-09-01T13:07:19.525960,2016-09-01T13:07:18.075960,2016-09-01T13:07:21.075960
2,Capsule pulse 1,capsule_impact,2016-09-01T13:07:24.420000,2016-09-01T13:07:28.432360,2016-09-01T13:07:28.312360,2016-09-01T13:07:28.762360
3,Capsule pulse 2,capsule_explosion,2016-09-01T13:07:24.987500,2016-09-01T13:07:28.999860,2016-09-01T13:07:28.869860,2016-09-01T13:07:29.269860


In [4]:
all_results = []
event_streams = []

window_rows = []

for spec in event_windows:
    event_stream, result = measure_reduced_pressures_in_window(
        st_corr,
        spec["start"],
        spec["end"],
        SENSOR_DISTANCES_M,
        reference_distance_m=1000.0,
        event_name=spec["event"],
        baseline_start_s=spec["baseline_start_s"],
        baseline_end_s=spec["baseline_end_s"],
        signal_start_s=spec["signal_start_s"],
        signal_end_s=spec["signal_end_s"],
    )

    event_streams.append(event_stream)
    all_results.append(result)

    window_rows.append({
        "event": spec["event"],
        "source_event_key": spec["source_event_key"],
        "source_time_utc": spec["source_time"].isoformat(),
        "source_time_epoch_s": float(spec["source_time"].timestamp),
        "arrival_reference_channel": REFERENCE_CHANNEL,
        "arrival_reference_distance_m": REFERENCE_CHANNEL_DISTANCE_M,
        "acoustic_propagation_speed_mps": ACOUSTIC_PROPAGATION_SPEED_MPS,
        "predicted_reference_travel_time_s": REFERENCE_CHANNEL_TRAVEL_TIME_S,
        "predicted_reference_arrival_utc": spec["predicted_reference_arrival"].isoformat(),
        "predicted_reference_arrival_epoch_s": float(spec["predicted_reference_arrival"].timestamp),
        "window_start_offset_from_predicted_arrival_s": spec[
            "window_start_offset_from_predicted_arrival_s"
        ],
        "window_end_offset_from_predicted_arrival_s": float(
            spec["end"] - spec["predicted_reference_arrival"]
        ),
        "signal_start_offset_from_predicted_arrival_s": float(
            spec["start"] + spec["signal_start_s"] - spec["predicted_reference_arrival"]
        ),
        "signal_end_offset_from_predicted_arrival_s": float(
            spec["start"] + spec["signal_end_s"] - spec["predicted_reference_arrival"]
        ),
        "window_start_utc": spec["start"].isoformat(),
        "window_end_utc": spec["end"].isoformat(),
        "window_start_epoch_s": float(spec["start"].timestamp),
        "window_end_epoch_s": float(spec["end"].timestamp),
        "window_duration_s": float(spec["end"] - spec["start"]),
        "baseline_start_s": float(spec["baseline_start_s"]),
        "baseline_end_s": float(spec["baseline_end_s"]),
        "signal_start_s": float(spec["signal_start_s"]),
        "signal_end_s": float(spec["signal_end_s"]),
        "pressure_reduction_reference_distance_m": 1000.0,
        "waveform_product": str(BASELINE_STREAM_FILE),
        "geometry_product": str(GEOMETRY_FILE),
    })

key_event_pressures = pd.concat(
    all_results,
    ignore_index=True,
)
key_event_windows = pd.DataFrame(window_rows)

pressure_output_file = (
    DERIVED_DIR / "key_event_pressure_measurements.csv"
)
window_output_file = (
    DERIVED_DIR / "key_event_measurement_windows.csv"
)
metadata_output_file = (
    DERIVED_DIR / "key_event_measurement_metadata.json"
)

key_event_pressures.to_csv(
    pressure_output_file,
    index=False,
)
key_event_windows.to_csv(
    window_output_file,
    index=False,
)

measurement_metadata = {
    "producer_notebook": "110_measure_named_events.ipynb",
    "analysis_configuration": str(ANALYSIS_CONFIG_FILE),
    "geometry_file": str(GEOMETRY_FILE),
    "waveform_product": str(BASELINE_STREAM_FILE),
    "pressure_channels": list(pressure_channels),
    "sensor_distances_m": {
        channel: float(distance)
        for channel, distance in SENSOR_DISTANCES_M.items()
    },
    "pressure_reduction_reference_distance_m": 1000.0,
    "arrival_reference_channel": REFERENCE_CHANNEL,
    "arrival_reference_distance_m": REFERENCE_CHANNEL_DISTANCE_M,
    "acoustic_propagation_speed_mps": ACOUSTIC_PROPAGATION_SPEED_MPS,
    "predicted_reference_travel_time_s": REFERENCE_CHANNEL_TRAVEL_TIME_S,
    "source_event_times_utc": {
        key: UTCDateTime(config.SOURCE_EVENT_TIMES[key]).isoformat()
        for key in required_source_events
    },
    "pressure_measurements_file": str(pressure_output_file),
    "measurement_windows_file": str(window_output_file),
}
metadata_output_file.write_text(
    json.dumps(measurement_metadata, indent=2) + "\n"
)

print("Key-event pressure measurements:")
display(pressure_results_for_paper(key_event_pressures))

print("Authoritative key-event measurement windows:")
display(key_event_windows)

print("Wrote:", pressure_output_file)
print("Wrote:", window_output_file)
print("Wrote:", metadata_output_file)


Key-event pressure measurements:


,event,channel,distance_m,positive_peak_pa,negative_peak_pa,peak_to_peak_pa,positive_reduced_pa,negative_reduced_pa,peak_to_peak_reduced_pa
0,Initial second-stage failure,DD1,1440.0,30.4,-72.2,102.6,43.8,-104.0,147.8
1,Initial second-stage failure,DD2,1408.3,51.7,-25.4,77.1,72.8,-35.8,108.6
2,Initial second-stage failure,DD3,1412.9,38.8,-21.2,60.1,54.9,-30.0,84.9
3,Initial second-stage failure,MEDIAN,1412.9,38.8,-25.4,77.1,54.9,-35.8,108.6
4,Principal explosion,DD1,1440.0,1309.1,-130.8,1439.9,1885.1,-188.4,2073.4
5,Principal explosion,DD2,1408.3,1512.4,-158.5,1670.9,2130.0,-223.2,2353.1
6,Principal explosion,DD3,1412.9,1392.3,-157.0,1549.4,1967.2,-221.9,2189.0
7,Principal explosion,MEDIAN,1412.9,1392.3,-157.0,1549.4,1967.2,-221.9,2189.0
8,Capsule pulse 1,DD1,1440.0,225.8,-128.1,353.8,325.1,-184.4,509.5
9,Capsule pulse 1,DD2,1408.3,279.1,-115.3,394.5,393.1,-162.4,555.5


Authoritative key-event measurement windows:


,event,source_event_key,source_time_utc,source_time_epoch_s,arrival_reference_channel,arrival_reference_distance_m,acoustic_propagation_speed_mps,predicted_reference_travel_time_s,predicted_reference_arrival_utc,predicted_reference_arrival_epoch_s,...,window_start_epoch_s,window_end_epoch_s,window_duration_s,baseline_start_s,baseline_end_s,signal_start_s,signal_end_s,pressure_reduction_reference_distance_m,waveform_product,geometry_product
0,Initial second-stage failure,upper_stage,2016-09-01T13:07:11.913000,1.472735e+09,DD2,1408.338457,351.0,4.01236,2016-09-01T13:07:15.925360,1.472735e+09,...,1.472735e+09,1.472735e+09,2.00,0.1,0.80,0.85,1.40,1000.0,/Users/thompsong/Developer/KSCRocketSeismology...,/Users/thompsong/Developer/KSCRocketSeismology...
1,Principal explosion,lower_stage,2016-09-01T13:07:15.513600,1.472735e+09,DD2,1408.338457,351.0,4.01236,2016-09-01T13:07:19.525960,1.472735e+09,...,1.472735e+09,1.472735e+09,3.00,0.1,0.75,0.75,2.50,1000.0,/Users/thompsong/Developer/KSCRocketSeismology...,/Users/thompsong/Developer/KSCRocketSeismology...
2,Capsule pulse 1,capsule_impact,2016-09-01T13:07:24.420000,1.472735e+09,DD2,1408.338457,351.0,4.01236,2016-09-01T13:07:28.432360,1.472735e+09,...,1.472735e+09,1.472735e+09,0.45,0.0,0.07,0.08,0.45,1000.0,/Users/thompsong/Developer/KSCRocketSeismology...,/Users/thompsong/Developer/KSCRocketSeismology...
3,Capsule pulse 2,capsule_explosion,2016-09-01T13:07:24.987500,1.472735e+09,DD2,1408.338457,351.0,4.01236,2016-09-01T13:07:28.999860,1.472735e+09,...,1.472735e+09,1.472735e+09,0.40,0.0,0.10,0.10,0.40,1000.0,/Users/thompsong/Developer/KSCRocketSeismology...,/Users/thompsong/Developer/KSCRocketSeismology...


Wrote: /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/falcon9-seismoacoustic-workflow-1.0.0/data/outputs/110_measure_named_events/key_event_pressure_measurements.csv
Wrote: /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/falcon9-seismoacoustic-workflow-1.0.0/data/outputs/110_measure_named_events/key_event_measurement_windows.csv
Wrote: /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/falcon9-seismoacoustic-workflow-1.0.0/data/outputs/110_measure_named_events/key_event_measurement_metadata.json
